In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchtext.vocab import build_vocab_from_iterator
from torchtext.vocab import Vocab
from torch import Tensor
from torch.nn import Transformer
from torchtext.data.utils import get_tokenizer
from torch.nn.utils.rnn import pad_sequence
from itertools import chain
from itertools import islice
from torchtext.datasets import IMDB
from copy import deepcopy
import random
import csv
import json
from tqdm import tqdm
import pandas as pd

# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [2]:
tokenizer=get_tokenizer("basic_english")

def yield_tokens(data_iter):
    for data_sample in data_iter:
        yield tokenizer(data_sample[1])
        
# Define special symbols and indices
PAD_IDX,CLS_IDX, SEP_IDX,  MASK_IDX,UNK_IDX= 0, 1, 2, 3, 4

# Make sure the tokens are in order of their indices to properly insert them in vocab
special_symbols = ['[PAD]','[CLS]', '[SEP]','[MASK]','[UNK]']

In [16]:
#create data splits
train_iter, test_iter = IMDB(split=('train', 'test'))
all_data_iter = chain(train_iter, test_iter)
#check tokenizer
fifth_item_tokens = next(islice(yield_tokens(all_data_iter), 6, None)) ## Start printing tokens from the 6th text
print(fifth_item_tokens)

['whoever', 'wrote', 'the', 'screenplay', 'for', 'this', 'movie', 'obviously', 'never', 'consulted', 'any', 'books', 'about', 'lucille', 'ball', ',', 'especially', 'her', 'autobiography', '.', 'i', "'", 've', 'never', 'seen', 'so', 'many', 'mistakes', 'in', 'a', 'biopic', ',', 'ranging', 'from', 'her', 'early', 'years', 'in', 'celoron', 'and', 'jamestown', 'to', 'her', 'later', 'years', 'with', 'desi', '.', 'i', 'could', 'write', 'a', 'whole', 'list', 'of', 'factual', 'errors', ',', 'but', 'it', 'would', 'go', 'on', 'for', 'pages', '.', 'in', 'all', ',', 'i', 'believe', 'that', 'lucille', 'ball', 'is', 'one', 'of', 'those', 'inimitable', 'people', 'who', 'simply', 'cannot', 'be', 'portrayed', 'by', 'anyone', 'other', 'than', 'themselves', '.', 'if', 'i', 'were', 'lucie', 'arnaz', 'and', 'desi', ',', 'jr', '.', ',', 'i', 'would', 'be', 'irate', 'at', 'how', 'many', 'mistakes', 'were', 'made', 'in', 'this', 'film', '.', 'the', 'filmmakers', 'tried', 'hard', ',', 'but', 'the', 'movie', 's

In [17]:
#create vocab : vocab is only built using train data
vocab=build_vocab_from_iterator(yield_tokens(all_data_iter),specials=special_symbols,special_first=True)

vocab.set_default_index(UNK_IDX)
VOCAB_SIZE=len(vocab)
print(VOCAB_SIZE)

98583


In [21]:
text_to_index=lambda text: [vocab(token) for token in tokenizer(text)] ## Pass text sample as input to retrieve token indices
index_to_en = lambda seq_en: " ".join([vocab.get_itos()[index] for index in seq_en]) ## Pass list of token_ids in a sequence as input to get back the original sentence 

In [28]:
# seq_en = [0, 1, 2, 3, 4, 5, 6]  # Example input sequence
# english_sentence = index_to_en(seq_en)
# seq2=[8385,232,768,21]
# english_sentence = index_to_en(seq2)

# print(english_sentence)

print(index_to_en([15, 175, 28, 5, 107, 28, 2535, 78, 43]))

text = "I love you. Will you marry me?"  # Example input text
text_to_index = lambda text: [vocab[token] for token in tokenizer(text)]
index_sequence = text_to_index(text)

print(index_sequence)

i love you . will you marry me ?
[15, 175, 28, 5, 107, 28, 2535, 78, 43]


In [43]:
def bernoulli_true_false(p): 
    bernoulli_dist = torch.distributions.Bernoulli(torch.tensor([p])) 
    return bernoulli_dist.sample().item() == 1

In [44]:
def Masking(token):
    
    mask = bernoulli_true_false(0.2)
    if not mask:
        return token, '[PAD]'
    
    random_opp = bernoulli_true_false(0.5)
    random_swich = bernoulli_true_false(0.5)
    
    if mask and random_opp and random_swich:
        mask_label = index_to_en(torch.randint(0, VOCAB_SIZE, (1,)))
        token_ = '[MASK]'

    elif mask and random_opp and not random_swich:
        token_ = token
        mask_label = token
    else:
        token_ = '[MASK]'
        mask_label = token

    return token_, mask_label

In [46]:

for l in range(50):
    token="apple"
    token_,label=Masking(token)
    if token==token_ and label=="[PAD]":
        print(token_,label,f"\t Actual token *{token}* is left unchanged")
    elif token_=="[MASK]" and label==token:
        print(token_,label,f"\t Actual token *{token}* is masked with '{token_}'")
    else:
        print(token_,label,f"\t Actual token *{token}* is replaced with random token #{label}#")

apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple apple 	 Actual token *apple* is replaced with random token #apple#
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
[MASK] apple 	 Actual token *apple* is masked with '[MASK]'
apple [PAD] 	 Actual token *apple* is left unchanged
apple [PAD] 	 Actual token *apple* is left unchanged
[MASK] colgate 	 Actual token *apple* is replaced with random token #col

In [47]:
def prepare_for_mlm(tokens, include_raw_tokens=False):
    """
    Prepares tokenized text for BERT's Masked Language Model (MLM) training.

    """
    bert_input = []  # List to store sentences processed for BERT's MLM
    bert_label = []  # List to store labels for each token (mask, random, or unchanged)
    raw_tokens_list = []  # List to store raw tokens if needed
    current_bert_input = []
    current_bert_label = []
    current_raw_tokens = []

    for token in tokens:
        # Apply BERT's MLM masking strategy to the token
        masked_token, mask_label = Masking(token)

        # Append the processed token and its label to the current sentence and label list
        current_bert_input.append(masked_token)
        current_bert_label.append(mask_label)

        # If raw tokens are to be included, append the original token to the current raw tokens list
        if include_raw_tokens:
            current_raw_tokens.append(token)

        # Check if the token is a sentence delimiter (., ?, !)
        if token in ['.', '?', '!']:
            # If current sentence has more than two tokens, consider it a valid sentence
            if len(current_bert_input) > 2:
                bert_input.append(current_bert_input)
                bert_label.append(current_bert_label)
                # If including raw tokens, add the current list of raw tokens to the raw tokens list
                if include_raw_tokens:
                    raw_tokens_list.append(current_raw_tokens)

                # Reset the lists for the next sentence
                current_bert_input = []
                current_bert_label = []
                current_raw_tokens = []
            else:
                # If the current sentence is too short, discard it and reset lists
                current_bert_input = []
                current_bert_label = []
                current_raw_tokens = []

    # Add any remaining tokens as a sentence if there are any
    if current_bert_input:
        bert_input.append(current_bert_input)
        bert_label.append(current_bert_label)
        if include_raw_tokens:
            raw_tokens_list.append(current_raw_tokens)

    # Return the prepared lists for BERT's MLM training
    return (bert_input, bert_label, raw_tokens_list) if include_raw_tokens else (bert_input, bert_label)

In [108]:
## Understanding BERT Data

# tokens=tokenizer('I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.')
# y=prepare_for_mlm(tokens,include_raw_tokens=True)
# print(len(y))
# d={'BERT Input':y[0][0],'BERT Label': y[1][0],'Original Token':y[2][0]}
# df=pd.DataFrame(d)
# df
## y[0] is BERT Input, y[1] is BERT Output

3


,BERT Input,BERT Label,Original Token
0,i,[PAD],i
1,rented,[PAD],rented
2,i,[PAD],i
3,am,[PAD],am
4,[MASK],curious-yellow,curious-yellow
5,from,[PAD],from
6,[MASK],my,my
7,video,video,video
8,store,[PAD],store
9,because,[PAD],because


In [109]:
def process_for_nsp(input_sentences, input_masked_labels):
    """
    Prepares data for Next Sentence Prediction (NSP) task in BERT training.

    Args:
    input_sentences (list): List of tokenized sentences.
    input_masked_labels (list): Corresponding list of masked labels for the sentences.

    Returns:
    bert_input (list): List of sentence pairs for BERT input.
    bert_label (list): List of masked labels for the sentence pairs.
    is_next (list): Binary label list where 1 indicates 'next sentence' and 0 indicates 'not next sentence'.
    """
    if len(input_sentences) < 2:
        raise ValueError("must have two same number of items.")
    ## Must have atleast two sentences for NSP to make sense


    # Verify that both input lists are of the same length and have a sufficient number of sentences
    if len(input_sentences) != len(input_masked_labels):
        raise ValueError("Both lists must have the same number of items.")

    bert_input = []
    bert_label = []
    is_next = []

    available_indices = list(range(len(input_sentences))) ## [0,1,2]

    while len(available_indices) >= 2:   ## 3 in our case
        if random.random() < 0.5:   ### Outputs in the interval [0, 1)
            # Choose two consecutive sentences to simulate the 'next sentence' scenario
            index = random.choice(available_indices[:-1])  # Exclude the last index, select any index xcept the last one
            # append list and add  '[CLS]' and  '[SEP]' tokens
            bert_input.append([['[CLS]']+input_sentences[index]+ ['[SEP]'],input_sentences[index + 1]+ ['[SEP]']])
            bert_label.append([['[PAD]']+input_masked_labels[index]+['[PAD]'], input_masked_labels[index + 1]+ ['[PAD]']])
            is_next.append(1)  # Label 1 indicates these sentences are consecutive

            # Remove the used indices
            available_indices.remove(index)
            if index + 1 in available_indices:  ## We created the dataset keeping in mind that the sentence next to the chose one as next sentence, but index+1 can be in available indices also, hence  we remove that too
                available_indices.remove(index + 1)
        else:
            # Choose two random distinct sentences to simulate the 'not next sentence' scenario
            indices = random.sample(available_indices, 2)
            bert_input.append([['[CLS]']+input_sentences[indices[0]]+['[SEP]'],input_sentences[indices[1]]+ ['[SEP]']])
            bert_label.append([['[PAD]']+input_masked_labels[indices[0]]+['[PAD]'], input_masked_labels[indices[1]]+['[PAD]']])
            is_next.append(0)  # Label 0 indicates these sentences are not consecutive

            # Remove the used indices
            available_indices.remove(indices[0])
            available_indices.remove(indices[1])



    return bert_input, bert_label, is_next

In [142]:
for i,j in enumerate(train_iter):
    x=j[1]
    break

## Sample sentences
input_sentences=[]
for j in [0,3,5,6]:
    input_sentences.append(x.split(".")[j])

In [148]:
flatten = lambda l: [item for sublist in l for item in sublist]
input_masked_labels=[]
for sentence in input_sentences:
    _, current_masked_label= prepare_for_mlm(sentence, include_raw_tokens=False)
    input_masked_labels.append(flatten(current_masked_label))

In [212]:
#flatten the tensor
flatten = lambda l: [item for sublist in l for item in sublist]

input_sentences = [["i", "love", "apples"], ["she", "enjoys", "reading", "books"], ["he", "likes", "playing", "guitar"]]

# Create masked labels for the sentences
input_masked_labels=[]
for sentence in input_sentences:
    _, current_masked_label= prepare_for_mlm(sentence, include_raw_tokens=False)
    input_masked_labels.append(flatten(current_masked_label))

# Create NSP pairs and labels
random.seed(100)
bert_input, bert_label, is_next = process_for_nsp(input_sentences, input_masked_labels)

# Print the output
print("BERT Input:")
for pair in bert_input:
    print(pair)
print("BERT Label:")
for pair in bert_label:
    print(pair)
print("Is Next: ", is_next)
print("-"*127)
random.seed(1000)
bert_input, bert_label, is_next = process_for_nsp(input_sentences, input_masked_labels)

# # Print the output
# print("BERT Input:")
# for pair in bert_input:
#     print(pair)
# print("BERT Label:")
# for pair in bert_label:
#     print(pair)
# print("Is Next: ", is_next)


BERT Input:
[['[CLS]', 'she', 'enjoys', 'reading', 'books', '[SEP]'], ['he', 'likes', 'playing', 'guitar', '[SEP]']]
BERT Label:
[['[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'], ['he', 'likes', 'playing', '[PAD]', '[PAD]']]
Is Next:  [1]
-------------------------------------------------------------------------------------------------------------------------------


In [214]:
def prepare_bert_final_inputs(bert_inputs, bert_labels, is_nexts,to_tenor=True):
    """
    Prepare the final input lists for BERT training.
    """
    def zero_pad_list_pair(pair_, pad='[PAD]'):
        pair=deepcopy(pair_)
        max_len = max(len(pair[0]), len(pair[1]))
        #append [PAD] to each sentence in the pair till the maximum length reaches
        pair[0].extend([pad] * (max_len - len(pair[0])))
        pair[1].extend([pad] * (max_len - len(pair[1])))
        return pair[0], pair[1]

    #flatten the tensor
    flatten = lambda l: [item for sublist in l for item in sublist]
    #transform tokens to vocab indices
    tokens_to_index=lambda tokens: [vocab[token] for token in tokens]

    bert_inputs_final, bert_labels_final, segment_labels_final, is_nexts_final = [], [], [], []

    for bert_input, bert_label,is_next in zip(bert_inputs, bert_labels,is_nexts):
        # Create segment labels for each pair of sentences
        segment_label = [[1] * len(bert_input[0]), [2] * len(bert_input[1])]

        # Zero-pad the bert_input and bert_label and segment_label
        bert_input_padded = zero_pad_list_pair(bert_input)
        bert_label_padded = zero_pad_list_pair(bert_label)
        segment_label_padded = zero_pad_list_pair(segment_label,pad=0)

        #convert to tensors
        if to_tenor:

            # Flatten the padded inputs and labels, transform tokens to their corresponding vocab indices, and convert them to tensors
            bert_inputs_final.append(torch.tensor(tokens_to_index(flatten(bert_input_padded)),dtype=torch.int64))
            #bert_labels_final.append(torch.tensor(tokens_to_index(flatten(bert_label_padded)),dtype=torch.int64))
            bert_labels_final.append(torch.tensor(tokens_to_index(flatten(bert_label_padded)),dtype=torch.int64))
            segment_labels_final.append(torch.tensor(flatten(segment_label_padded),dtype=torch.int64))
            is_nexts_final.append(is_next)

        else:
          # Flatten the padded inputs and labels
            bert_inputs_final.append(flatten(bert_input_padded))
            bert_labels_final.append(flatten(bert_label_padded))
            segment_labels_final.append(flatten(segment_label_padded))
            is_nexts_final.append(is_next)

    return bert_inputs_final, bert_labels_final, segment_labels_final, is_nexts_final


In [215]:
bert_inputs_final, bert_labels_final, segment_labels_final, is_nexts_final=prepare_bert_final_inputs(bert_input, bert_label, is_next,to_tenor=True)
torch.set_printoptions(linewidth=10000)# this assures that whole output is printed in one line
print("input:\t\t",bert_input,"\ninputs_final:\t",bert_inputs_final,"\nbert labels final:\t",bert_labels_final,"\nsegment labels final:\t",segment_labels_final,"\nis nexts final:\t",is_nexts_final)

input:		 [[['[CLS]', 'he', 'likes', 'playing', 'guitar', '[SEP]'], ['i', 'love', 'apples', '[SEP]']]] 
inputs_final:	 [tensor([    1,    36,  1148,   431,  5122,     2,    15,   175, 13934,     2,     0,     0])] 
bert labels final:	 [tensor([    0,    36,  1148,   431,     0,     0,     0,     0, 13934,     0,     0,     0])] 
segment labels final:	 [tensor([1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 0, 0])] 
is nexts final:	 [0]


In [225]:
train_iter

ShardingFilterIterDataPipe

##### Tokens 'he'= 36, 'likes'=1148,'playing'=431,'apples'=13934 are masked

In [226]:
csv_file_path ='train_bert_data_new.csv'
with open(csv_file_path, mode='w', newline='', encoding='utf-8') as file:
    csv_writer = csv.writer(file)
    csv_writer.writerow(['Original Text', 'BERT Input', 'BERT Label', 'Segment Label', 'Is Next'])

    # Wrap train_iter with tqdm for a progress bar
    for n, (_, sample) in enumerate(tqdm(train_iter, desc="Processing samples")):
        # Tokenize the sample input
        tokens = tokenizer(sample)
        # Create MLM inputs and labels
        bert_input, bert_label = prepare_for_mlm(tokens, include_raw_tokens=False)
        if len(bert_input) < 2:
            continue
        # Create NSP pairs, token labels, and is_next label
        bert_inputs, bert_labels, is_nexts = process_for_nsp(bert_input, bert_label)
        # add zero-paddings, map tokens to vocab indices and create segment labels
        bert_inputs, bert_labels, segment_labels, is_nexts = prepare_bert_final_inputs(bert_inputs, bert_labels, is_nexts)
        # convert tensors to lists, convert lists to JSON-formatted strings
        for bert_input, bert_label, segment_label, is_next in zip(bert_inputs, bert_labels, segment_labels, is_nexts):
            bert_input_str = json.dumps(bert_input.tolist())
            bert_label_str = json.dumps(bert_label.tolist())
            segment_label_str = ','.join(map(str, segment_label.tolist()))
            # Write the data to a CSV file row-by-row
            csv_writer.writerow([sample, bert_input_str, bert_label_str, segment_label_str, is_next])
        if n>5:
            break

Processing samples: 6it [00:02,  2.83it/s]
